# Exploring historical pathogens

- **Summary**: This document explores the historical occurrences and severities of existing viral pathogens

- **Data Required:** `epidemics_marani_240816.xlsx`

- **Author**: Ganqi Li (ganqi.li.25@dartmouth.edu)
- **Updated**: Sep 14, 2024

In [63]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.integrate import quad
from scipy.stats import genpareto
from scipy.optimize import minimize

## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## 0. Prepare respiratory viral families

In [64]:
## Read epidemic data from Marani et al. 
df = pd.read_excel("data/epidemics_marani_240816.xlsx")
df = df.sort_values(by='year_start', ascending=True).reset_index(drop=True)


In [65]:
## Explore unique viral diseases
viral_diseases = df[df['type'].str.contains("viral", case=False, na=False)]['disease'].unique()

## Display results
print(f"{len(viral_diseases)} symptoms that may be caused by viral pathogens:")
print()
print(viral_diseases)


22 symptoms that may be caused by viral pathogens:

['smallpox' 'influenza' 'measles' 'yellow fever' 'rubella'
 'venereal disease' 'meningitis' 'dengue' 'polio' 'pneumonia'
 'murray valley encephalitis' 'encephalitis' 'mumps' 'west nile'
 'kyasanur forest disease' 'hemorrhagic fever' 'ebola' 'rift valley fever'
 'hiv/aids' 'sars' 'mers' 'covid-19']


In [66]:
## Map viral symptoms to families
family_mapping = {
    'small pox': 'poxviridae',
    'influenza': 'orthomyxoviridae',
    'measles': 'paramyxoviridae',
    'yellow fever': 'flaviviridae',
    'rubella': 'matonaviridae',
    'venereal disease': 'misc',
    'meningitis': 'misc',
    'dengue': 'flaviviridae',
    'polio': 'picornaviridae',
    'pneumonia': 'misc',
    'murray valley encephalitis': 'flaviviridae',
    'encephalitis': 'misc',
    'mumps': 'paramyxoviridae',
    'west nile': 'flaviviridae',
    'kyasanur forest disease': 'flaviviridae',
    'hemorrhagic fever': 'misc',
    'ebola': 'filoviridae',
    'rift valley fever': 'bunyaviridae',
    'hiv/aids': 'retroviridae',
    'sars': 'coronaviridae',
    'mers': 'coronaviridae',
    'covid-19': 'coronaviridae',
}

## Create family column 
df['family'] = df['disease'].map(family_mapping)


In [69]:
## Save dataframe to excel file
# df.to_excel("data/epidemics_marani_240914_family.xlsx", index=False)


## 1. Explore viral epidemics in the modern period

In [70]:
## Subset data to specific modern period
yrmin = 1500
df = df[(df["year_start"] >= yrmin)].reset_index(drop=True)


In [71]:
## Subset to viral epidemics
df_viral = df[df['type'].str.contains('viral', case=False, na=False)
             ].reset_index(drop=True)

## Subset to viral epidemics with known intensities
df_viral_known = df_viral[df_viral["severity_smu"] > 0].reset_index(drop=True)


In [72]:
## Group occurrence by viral families
df_viral_unique = df_viral_known.groupby(['family']
                                        ).size().reset_index(name='count')

## Sort by number of occurrences in descending order
df_viral_unique = df_viral_unique.sort_values(by='count', 
                                              ascending=False).reset_index(drop=True)


In [73]:
df_viral_unique 

,family,count
0,flaviviridae,23
1,misc,14
2,orthomyxoviridae,11
3,paramyxoviridae,8
4,picornaviridae,6
5,coronaviridae,3
6,filoviridae,3
7,matonaviridae,1
8,retroviridae,1


In [40]:
## Label sars, mers, and covid-19 as coronavirus
df.loc[df['disease'].isin(['covid-19', 'mers', 'sars']), 'disease'] = 'coronavirus'


In [41]:
## Subset data to specific modern period
yrmin = 1500
df = df[(df["year_start"] >= yrmin)].reset_index(drop=True)


In [42]:
## Subset to viral epidemics
df_viral = df[df['type'].str.contains('viral', case=False, na=False)
             ].reset_index(drop=True)

## Subset to viral epidemics with known intensities
df_viral_known = df_viral[df_viral["severity_smu"] > 0].reset_index(drop=True)


**Note:** We may have to modify Madhav's exceedance probabilities by scaling the respiratory exceedance upward to an all-viral exceedance, and then apply the distribution of the expected average annual loss (AAL) of each viral pathogen.

In [43]:
## Group occurrence by viral pathogens
df_viral_unique = df_viral_known.groupby(['disease', 'type']
                                        ).size().reset_index(name='count')

## Sort by number of occurrences in descending order
df_viral_unique = df_viral_unique.sort_values(by='count', 
                                              ascending=False).reset_index(drop=True)


In [44]:
## Specify viral families
# df_viral_unique['family_1600'] = ['poxviridae', 'flaviviridae', 'orthomyxoviridae', 
#                                   'paramyxoviridae', 'misc', 'picornaviridae', 
#                                   'misc', 'coronaviridae', 'flaviviridae', 'filoviridae', 
#                                   'misc', 'misc', 'retroviridae', 'matonaviridae']





## Display
print()
print(f'Epidemics {yrmin}-present:')
df_viral_unique



Epidemics 1500-present:


,disease,type,count
0,smallpox,viral,48
1,yellow fever,viral,21
2,influenza,viral,11
3,measles,viral,8
4,meningitis,bacterial/viral,7
5,polio,viral,6
6,encephalitis,bacterial/fungal/parasitic/viral,4
7,coronavirus,viral,3
8,ebola,viral,3
9,dengue,viral,2


In [46]:
df_viral_known[df_viral_known['disease'] == 'influenza']

,location,year_start,year_end,duration,death_thousand,pop_thousand,severity_perthousand,severity_smu,disease,type,transmission,is_vira_only,is_vira_mixed,contains_vira,is_pandemic
22,"pandemic, influenza",1781,1782,2,100.000,8.698400e+05,0.114964,1.149637,influenza,viral,droplet,1,0,1,1
39,west washington influenza,1836,1837,2,8.000,1.147067e+06,0.006974,0.069743,influenza,viral,droplet,1,0,1,0
63,russian flu,1889,1890,2,1310.000,1.497480e+06,0.874803,8.748030,influenza,viral,droplet,1,0,1,1
69,"usa, yukon",1900,1900,1,0.100,1.633000e+06,0.000061,0.000612,influenza,viral,droplet,1,0,1,0
76,pandemic spanish flu,1918,1920,3,32000.000,1.873300e+06,17.082154,170.821545,influenza,viral,droplet,1,0,1,1
81,"salomon islands, ontong java",1926,1926,1,0.033,2.020467e+06,0.000016,0.000163,influenza,viral,droplet,1,0,1,0
98,pandemic of asian flu,1957,1958,2,2000.000,2.873306e+06,0.696062,6.960623,influenza,viral,droplet,1,0,1,1
102,hong kong flu,1968,1969,2,1000.000,3.551599e+06,0.281563,2.815633,influenza,viral,droplet,1,0,1,1
103,papua new guinea,1969,1970,2,3.000,3.625681e+06,0.000827,0.008274,influenza,viral,droplet,1,0,1,0
111,swine flu,2009,2009,1,284.500,6.872767e+06,0.041395,0.413953,influenza,viral,droplet,1,0,1,1
